# Eval V2 — Context-Aware Outfit Recommendation

Runs 5 pipeline variants across 2 prompt contexts on the full 465-item wardrobe.
Results are saved to `eval_v2/results/` for human annotation via `eval_v2/annotate.html`.

| Pipeline | Retrieval | LLM Input | LLM? |
|---|---|---|---|
| P1 text_llm | None (all 465 items) | Full metadata as text | Yes |
| P2 clip_only | FashionCLIP top-k | — | No |
| P3 clip_text_llm | FashionCLIP top-k | Text metadata of candidates | Yes |
| P4 clip_vision_llm | FashionCLIP top-k | Candidate images | Yes (vision) |
| P5 filtered_text_llm | Metadata filter (season+occasion) | Filtered metadata as text | Yes |

**Run order:** Sections 1–7 once per session, then Section 8 (Config), then Section 9 (Run).
After running, open `eval_v2/annotate.html` for human scoring.

## 1. Imports

In [47]:
from __future__ import annotations

import base64
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import torch
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

load_dotenv()
print('Imports OK')

Imports OK


## 2. Project paths

In [48]:
def _project_root() -> Path:
    here = Path.cwd().resolve()
    if here.name == 'notebooks':
        return here
    if (here / 'notebooks').is_dir():
        return here / 'notebooks'
    return here

ROOT        = _project_root()
EVAL_DIR    = ROOT / 'eval_polyvore'          # existing 465-item wardrobe
IMAGES_DIR  = EVAL_DIR / 'images'
META_DIR    = EVAL_DIR / 'metadata'
CACHE_PATH  = EVAL_DIR / 'embeddings.npz'     # reuse existing combined embeddings if present
MODEL_DIR   = ROOT / 'models' / 'fashion-clip'
RESULTS_DIR = ROOT / 'eval_v2' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'ROOT:        {ROOT}')
print(f'IMAGES_DIR:  {IMAGES_DIR}  (exists={IMAGES_DIR.exists()})')
print(f'META_DIR:    {META_DIR}  (exists={META_DIR.exists()})')
print(f'RESULTS_DIR: {RESULTS_DIR}')

ROOT:        /Users/mohitjoshi/Projects/fashion-app/notebooks
IMAGES_DIR:  /Users/mohitjoshi/Projects/fashion-app/notebooks/eval_polyvore/images  (exists=True)
META_DIR:    /Users/mohitjoshi/Projects/fashion-app/notebooks/eval_polyvore/metadata  (exists=True)
RESULTS_DIR: /Users/mohitjoshi/Projects/fashion-app/notebooks/eval_v2/results


## 3. Load wardrobe metadata

In [49]:
def load_wardrobe(meta_dir: Path) -> dict[str, dict]:
    """Returns {item_id (no ext): metadata} for all JSON files."""
    wardrobe: dict[str, dict] = {}
    for p in sorted(meta_dir.glob('*.json')):
        try:
            meta = json.loads(p.read_text())
            item_id = p.stem  # e.g. '182455006'
            meta['item_id'] = item_id
            wardrobe[item_id] = meta
        except Exception as e:
            print(f'  skip {p.name}: {e}')
    return wardrobe


def metadata_to_text(meta: dict) -> str:
    """Compact one-line description for text embedding."""
    colors   = ', '.join(meta.get('colors', {}).get('all', []))
    tags     = ', '.join(meta.get('style', {}).get('style_tags', []))
    occ      = ', '.join(meta.get('style', {}).get('occasion_suitability', []))
    seasons  = ', '.join(meta.get('seasonality', {}).get('season', []))
    fit      = meta.get('shape_fit', {}).get('fit', '')
    return (
        f"type: {meta.get('type','')}; category: {meta.get('category','')}; "
        f"colors: {colors}; style: {tags}; fit: {fit}; "
        f"occasion: {occ}; season: {seasons}"
    )


wardrobe = load_wardrobe(META_DIR)

from collections import Counter
cat_counts = Counter(m['category'] for m in wardrobe.values())
VALID_CATS = {'tops', 'bottoms', 'outerwear'}

print(f'Loaded {len(wardrobe)} items')
print('Category breakdown:', dict(cat_counts))

# item_id → filename (with .jpg)
id_to_file = {item_id: item_id + '.jpg' for item_id in wardrobe}

Loaded 465 items
Category breakdown: {'bottoms': 199, 'outerwear': 73, 'tops': 189, 'other': 2, 'dress': 2}


## 4. FashionCLIP model + embedding helpers

In [50]:
def load_fashionclip(
    model_name: str = 'patrickjohncyh/fashion-clip',
    local_dir: Path | None = None,
) -> tuple[CLIPModel, CLIPProcessor, torch.device]:
    device = torch.device(
        'mps'  if torch.backends.mps.is_available()  else
        'cuda' if torch.cuda.is_available()           else 'cpu'
    )
    local_dir = local_dir or MODEL_DIR
    source = str(local_dir) if local_dir.exists() else model_name
    processor = CLIPProcessor.from_pretrained(source)
    model     = CLIPModel.from_pretrained(source).to(device).eval()
    if source == model_name:
        local_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(local_dir))
        processor.save_pretrained(str(local_dir))
    print(f'FashionCLIP ready on {device}')
    return model, processor, device


def _img_feats(m, pv):  return m.visual_projection(m.vision_model(pixel_values=pv).pooler_output)
def _txt_feats(m, ids, mask): return m.text_projection(m.text_model(input_ids=ids, attention_mask=mask).pooler_output)


def embed_images(paths: list[Path], *, model, processor, device, batch=16) -> np.ndarray:
    vecs = []
    for i in range(0, len(paths), batch):
        imgs = [Image.open(p).convert('RGB') for p in paths[i:i+batch]]
        pv   = processor(images=imgs, return_tensors='pt')['pixel_values'].to(device)
        with torch.no_grad():
            f = _img_feats(model, pv)
        f = f / f.norm(dim=-1, keepdim=True)
        vecs.append(f.cpu().float().numpy())
    return np.concatenate(vecs)


def embed_texts(texts: list[str], *, model, processor, device, batch=64) -> np.ndarray:
    vecs = []
    for i in range(0, len(texts), batch):
        enc  = processor(text=texts[i:i+batch], return_tensors='pt', padding=True, truncation=True)
        ids  = enc['input_ids'].to(device)
        mask = enc['attention_mask'].to(device)
        with torch.no_grad():
            f = _txt_feats(model, ids, mask)
        f = f / f.norm(dim=-1, keepdim=True)
        vecs.append(f.cpu().float().numpy())
    return np.concatenate(vecs)


def embed_query(query: str, *, model, processor, device) -> np.ndarray:
    return embed_texts([query], model=model, processor=processor, device=device)

## 5. Build / load embeddings

In [51]:
def build_embeddings(
    wardrobe: dict[str, dict],
    images_dir: Path,
    cache_path: Path,
    *,
    model, processor, device,
    alpha: float = 0.5,   # image weight; 1-alpha = metadata text weight
    force: bool = False,
) -> tuple[np.ndarray, list[str]]:
    """Returns (embeddings, item_ids). Combined image+text embedding per item."""
    if not force and cache_path.is_file():
        data = np.load(cache_path, allow_pickle=False)
        id_key = 'ids' if 'ids' in data else 'filenames'  # handle both cache formats
        print(f'Loaded embedding cache: {len(data[id_key])} items  ({cache_path.name})')
        return data['embeddings'], data[id_key].tolist()

    item_ids = [iid for iid in wardrobe if (images_dir / (iid + '.jpg')).exists()]
    print(f'Embedding {len(item_ids)} items (alpha={alpha})...')

    paths      = [images_dir / (iid + '.jpg') for iid in item_ids]
    img_vecs   = embed_images(paths, model=model, processor=processor, device=device)

    texts      = [metadata_to_text(wardrobe[iid]) for iid in item_ids]
    text_vecs  = embed_texts(texts, model=model, processor=processor, device=device)

    combined   = alpha * img_vecs + (1 - alpha) * text_vecs
    norms      = np.linalg.norm(combined, axis=-1, keepdims=True)
    combined  /= np.where(norms > 0, norms, 1.0)

    np.savez(cache_path, embeddings=combined, ids=np.array(item_ids))
    print(f'Saved -> {cache_path}')
    return combined, item_ids


clip_model, clip_proc, clip_device = load_fashionclip()

# FORCE_REEMBED = True  # uncomment if metadata changed
embeddings, emb_ids = build_embeddings(
    wardrobe, IMAGES_DIR, CACHE_PATH,
    model=clip_model, processor=clip_proc, device=clip_device,
    force=False,
)
print(f'Embeddings shape: {embeddings.shape}  ({len(emb_ids)} items)')


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

FashionCLIP ready on mps
Loaded embedding cache: 465 items  (embeddings.npz)
Embeddings shape: (465, 512)  (465 items)


## 6. LLM client + cost helpers

In [67]:
# Gemini via OpenRouter: $0.25/M input tokens, $1.50/M output tokens
COST_INPUT_PER_M  = 0.25
COST_OUTPUT_PER_M = 1.50


def make_client() -> OpenAI:
    key = os.environ.get('OPENROUTER_API_KEY', '')
    if not key:
        raise EnvironmentError('OPENROUTER_API_KEY not set in .env')
    return OpenAI(api_key=key, base_url='https://openrouter.ai/api/v1')


def calc_cost(usage) -> dict:
    """Extract token counts and compute cost from an OpenRouter response.usage object."""
    inp  = getattr(usage, 'prompt_tokens',     0) or 0
    out  = getattr(usage, 'completion_tokens', 0) or 0
    cost = (inp * COST_INPUT_PER_M + out * COST_OUTPUT_PER_M) / 1_000_000
    return {'input_tokens': inp, 'output_tokens': out, 'cost_usd': round(cost, 6)}


def parse_json_response(text: str) -> dict:
    """Strip markdown fences and parse JSON from LLM response."""
    text = re.sub(r'^```[\w]*\n?', '', text.strip())
    text = re.sub(r'\n?```$', '', text.strip())
    return json.loads(text)


def validate_outfits(outfits: list[dict], wardrobe: dict) -> list[dict]:
    """Keep only outfits where all item_ids exist in wardrobe and have correct categories."""
    valid = []
    used  = set()
    for outfit in outfits:
        ok = True
        for cat, item_id in outfit.items():
            item_id = str(item_id).strip('[]').strip()  # coerce int→str, strip label brackets
            outfit[cat] = item_id
            if item_id not in wardrobe:
                print(f'  Warning: {item_id} not in wardrobe — skipping outfit')
                ok = False; break
            if wardrobe[item_id].get('category') != cat:
                print(f'  Warning: {item_id} is {wardrobe[item_id].get("category")} not {cat} — skipping')
                ok = False; break
            if item_id in used:
                print(f'  Warning: {item_id} reused across outfits — skipping')
                ok = False; break
        if ok:
            used.update(outfit.values())
            valid.append(outfit)
    return valid[:5]


llm_client = make_client()
print('LLM client ready')

LLM client ready


## 7. Retrieval helpers (shared by P2, P3, P4)

In [68]:
def retrieve_top_k(
    query: str,
    *,
    categories: list[str] | None = None,
    k: int = 10,
) -> dict[str, list[str]]:
    """Returns {category: [item_id, ...]} top-k per category by CLIP similarity."""
    q_vec  = embed_query(query, model=clip_model, processor=clip_proc, device=clip_device)
    scores = (embeddings @ q_vec.T).squeeze(-1)   # (N,)

    cats = categories or list(VALID_CATS)
    result: dict[str, list[str]] = {}
    for cat in cats:
        idx  = [i for i, iid in enumerate(emb_ids) if wardrobe.get(iid, {}).get('category') == cat]
        ranked = sorted(idx, key=lambda i: scores[i], reverse=True)[:k]
        result[cat] = [emb_ids[i] for i in ranked]
    return result


def format_candidates_text(candidates_by_cat: dict[str, list[str]]) -> str:
    """Format candidate items as structured text for the LLM prompt."""
    lines = []
    for cat, item_ids in candidates_by_cat.items():
        lines.append(f'\n### {cat.upper()} candidates')
        for iid in item_ids:
            meta  = wardrobe.get(iid, {})
            colors  = ', '.join(meta.get('colors', {}).get('all', []))
            tags    = ', '.join(meta.get('style', {}).get('style_tags', []))
            occ     = ', '.join(meta.get('style', {}).get('occasion_suitability', []))
            seasons = ', '.join(meta.get('seasonality', {}).get('season', []))
            pairing = ', '.join(meta.get('pairing_hints', {}).get('goes_well_with', []))
            lines.append(
                f'  - {iid}: {meta.get("type","?")} | colors: {colors} | '
                f'style: {tags} | occasion: {occ} | season: {seasons} | pairs_with: {pairing}'
            )
    return '\n'.join(lines)


OUTFIT_JSON_SCHEMA = (
    '{"outfits": ['
    '{"tops": "item_id", "bottoms": "item_id"},  '
    '{"tops": "item_id", "bottoms": "item_id", "outerwear": "item_id"}, '
    '{"tops": "item_id", "bottoms": "item_id"}]}'
)

STYLIST_SYSTEM = 'You are an expert fashion stylist. Return only valid JSON, no markdown.'


def diversity_instruction() -> str:
    return (
        'Provide exactly 5 DIVERSE outfit combinations. '
        'Each outfit must use completely different items — no item_id may appear in more than one outfit. '
        'Each outfit needs one top and one bottom; add outerwear only if it suits the occasion and season.'
    )

---
## 8. Config — edit here

Change `LLM_MODEL` and `CANDIDATES_PER_CAT` if needed. Everything else is auto-driven by `run_config.json`.

In [69]:
LLM_MODEL           = 'google/gemini-3.1-flash-lite-preview'   # change to your preferred Gemini model on OpenRouter
CANDIDATES_PER_CAT  = 12    # top-k per category for P3 / P5 text pipelines
VISION_CANDIDATES   = 6     # top-k per category for P4 (fewer: each image costs tokens)
NUM_RECS            = 5     # recommendations per pipeline per run
FORCE_RERUN         = False  # set True to overwrite existing result files

with open(ROOT / 'eval_v2' / 'run_config.json') as f:
    config = json.load(f)

runs      = config['runs']
pipelines = [p['id'] for p in config['pipelines']]
print('Runs:', [r['run_id'] for r in runs])
print('Pipelines:', pipelines)

Runs: ['r1_office_winter', 'r2_party_summer']
Pipelines: ['P1_text_llm', 'P2_clip_only', 'P3_clip_text_llm', 'P4_clip_vision_llm', 'P5_filtered_text_llm', 'P6_filtered_vision_llm']


## 9. Pipeline implementations

In [70]:
# ── P1: text_llm ──────────────────────────────────────────────────────────────
# Sends all 465 items' metadata as compact text to LLM. No visual retrieval.

def run_p1_text_llm(run: dict) -> dict:
    occasion, season = run['occasion'], run['season']

    # Compact one-liner per item — keeps prompt size manageable
    lines = []
    for iid, meta in wardrobe.items():
        if meta.get('category') not in VALID_CATS:
            continue
        colors   = ', '.join(meta.get('colors', {}).get('all', []))
        tags     = ', '.join(meta.get('style', {}).get('style_tags', []))
        occ      = ', '.join(meta.get('style', {}).get('occasion_suitability', []))
        seasons  = ', '.join(meta.get('seasonality', {}).get('season', []))
        lines.append(
            f'{iid} [{meta["category"]}] {meta.get("type","?")} | {colors} | {tags} | occ: {occ} | seasons: {seasons}'
        )
    wardrobe_text = '\n'.join(lines)

    user = (
        f'User request: occasion="{occasion}", season="{season}"\n\n'
        f'WARDROBE ({len(lines)} items):\n{wardrobe_text}\n\n'
        f'{diversity_instruction()}\n'
        f'Use only item_ids from the wardrobe above.\n'
        f'Return JSON: {OUTFIT_JSON_SCHEMA}\n'
        f'Provide {NUM_RECS + 2} outfits (extra buffer in case some are filtered); I will use the first {NUM_RECS} valid ones.'
    )

    t0 = time.perf_counter()
    resp = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'system', 'content': STYLIST_SYSTEM}, {'role': 'user', 'content': user}],
        temperature=0.4,
    )
    latency = time.perf_counter() - t0

    outfits = parse_json_response(resp.choices[0].message.content).get('outfits', [])
    outfits = validate_outfits(outfits, wardrobe)

    return {'recommendations': outfits, 'latency_s': round(latency, 3), **calc_cost(resp.usage)}

In [71]:
# ── P2: clip_only ─────────────────────────────────────────────────────────────
# FashionCLIP text→image similarity, greedy diverse selection. No LLM.

def run_p2_clip_only(run: dict) -> dict:
    occasion, season = run['occasion'], run['season']
    query = f'Season: {season}, Occasion: {occasion}, stylish complete outfit'

    t0     = time.perf_counter()
    q_vec  = embed_query(query, model=clip_model, processor=clip_proc, device=clip_device)
    scores = (embeddings @ q_vec.T).squeeze(-1)

    # Per-category sorted lists
    cat_ranked: dict[str, list[str]] = {}
    for cat in ['tops', 'bottoms', 'outerwear']:
        idx = [
            (i, float(scores[i])) for i, iid in enumerate(emb_ids)
            if wardrobe.get(iid, {}).get('category') == cat
        ]
        cat_ranked[cat] = [emb_ids[i] for i, _ in sorted(idx, key=lambda x: x[1], reverse=True)]

    # Greedy diverse: best combo → remove items → repeat
    used, outfits = set(), []
    for _ in range(NUM_RECS):
        outfit: dict[str, str] = {}
        for cat in ['tops', 'bottoms', 'outerwear']:
            for iid in cat_ranked.get(cat, []):
                if iid not in used:
                    outfit[cat] = iid
                    used.add(iid)
                    break
        if 'tops' in outfit and 'bottoms' in outfit:
            outfits.append(outfit)

    latency = time.perf_counter() - t0
    return {'recommendations': outfits, 'latency_s': round(latency, 3),
            'input_tokens': 0, 'output_tokens': 0, 'cost_usd': 0.0}

In [72]:
# ── P3: clip_text_llm ─────────────────────────────────────────────────────────
# FashionCLIP retrieves top-k per category. LLM selects from their text metadata.

def run_p3_clip_text_llm(run: dict) -> dict:
    occasion, season = run['occasion'], run['season']
    query = f'Season: {season}, Occasion: {occasion}, stylish complete outfit'

    t0 = time.perf_counter()
    candidates = retrieve_top_k(query, k=CANDIDATES_PER_CAT)
    candidate_text = format_candidates_text(candidates)

    user = (
        f'User request: occasion="{occasion}", season="{season}"\n'
        f'Pre-filtered candidates (ranked by visual similarity to query):\n'
        f'{candidate_text}\n\n'
        f'{diversity_instruction()}\n'
        f'Use only item_ids from the candidates above.\n'
        f'Return JSON: {OUTFIT_JSON_SCHEMA}\n'
        f'Provide {NUM_RECS + 2} outfits (extra buffer); I will use the first {NUM_RECS} valid ones.'
    )

    resp = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'system', 'content': STYLIST_SYSTEM}, {'role': 'user', 'content': user}],
        temperature=0.4,
    )
    latency = time.perf_counter() - t0

    outfits = parse_json_response(resp.choices[0].message.content).get('outfits', [])
    outfits = validate_outfits(outfits, wardrobe)
    return {'recommendations': outfits, 'latency_s': round(latency, 3), **calc_cost(resp.usage)}

In [73]:
# ── P4: clip_vision_llm ───────────────────────────────────────────────────────
# FashionCLIP retrieves top-k per category. Vision LLM sees the actual images.

def _encode_image(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode()


def run_p4_clip_vision_llm(run: dict) -> dict:
    occasion, season = run['occasion'], run['season']
    query = f'Season: {season}, Occasion: {occasion}, stylish complete outfit'

    t0 = time.perf_counter()
    candidates = retrieve_top_k(query, k=VISION_CANDIDATES)

    # Build multimodal message content: text intro + images per category
    content: list[dict] = [{
        'type': 'text',
        'text': (
            f'User request: occasion="{occasion}", season="{season}".\n'
            f'Below are wardrobe candidates per category, pre-filtered by visual similarity. '
            f'Each image is labelled with its item_id.\n'
        )
    }]

    for cat, item_ids in candidates.items():
        content.append({'type': 'text', 'text': f'\n### {cat.upper()} candidates:'})
        for iid in item_ids:
            img_path = IMAGES_DIR / (iid + '.jpg')
            if not img_path.exists():
                continue
            content.append({
                'type': 'image_url',
                'image_url': {'url': f'data:image/jpeg;base64,{_encode_image(img_path)}'}
            })
            content.append({'type': 'text', 'text': f'id:{iid}'})

    content.append({'type': 'text', 'text': (
        f'\n{diversity_instruction()}\n'
        f'Use only item_ids shown in the labels above (e.g. id:182455006 → use 182455006).\n'
        f'Return JSON: {OUTFIT_JSON_SCHEMA}\n'
        f'Provide {NUM_RECS + 2} outfits (extra buffer); I will use the first {NUM_RECS} valid ones.'
    )})

    resp = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'system', 'content': STYLIST_SYSTEM}, {'role': 'user', 'content': content}],
        temperature=0.4,
    )
    latency = time.perf_counter() - t0

    outfits = parse_json_response(resp.choices[0].message.content).get('outfits', [])
    outfits = validate_outfits(outfits, wardrobe)
    return {'recommendations': outfits, 'latency_s': round(latency, 3), **calc_cost(resp.usage)}

In [74]:
# ── P5: filtered_text_llm ─────────────────────────────────────────────────────
# Metadata pre-filter by season + occasion keywords, then LLM selects from filtered set.
# Tests: structured semantic filtering vs visual retrieval (compare with P3).

SEASON_CLUSTERS = {
    'winter': ['winter', 'fall', 'autumn'],
    'summer': ['summer', 'spring'],
}


def filter_by_context(occasion: str, season: str) -> dict[str, dict]:
    """Keep items that match on season OR occasion. Falls back to full wardrobe if too few."""
    relevant_seasons = SEASON_CLUSTERS.get(season.lower(), [season.lower()])
    occ_keywords     = set(re.split(r'\W+', occasion.lower()))

    filtered: dict[str, dict] = {}
    for iid, meta in wardrobe.items():
        if meta.get('category') not in VALID_CATS:
            continue
        item_seasons  = [s.lower() for s in meta.get('seasonality', {}).get('season', [])]
        item_occ      = ' '.join(meta.get('style', {}).get('occasion_suitability', [])).lower()

        season_match  = any(s in relevant_seasons for s in item_seasons)
        occ_match     = any(kw in item_occ for kw in occ_keywords if len(kw) > 3)

        if season_match or occ_match:
            filtered[iid] = meta

    # Ensure at least one item per category — fallback: add all-season items
    cats_found = {m['category'] for m in filtered.values()}
    for cat in VALID_CATS:
        if cat not in cats_found:
            for iid, meta in wardrobe.items():
                if meta.get('category') == cat:
                    filtered[iid] = meta
    return filtered


def run_p5_filtered_text_llm(run: dict) -> dict:
    occasion, season = run['occasion'], run['season']

    t0 = time.perf_counter()
    filtered = filter_by_context(occasion, season)

    lines = []
    for iid, meta in filtered.items():
        colors  = ', '.join(meta.get('colors', {}).get('all', []))
        tags    = ', '.join(meta.get('style', {}).get('style_tags', []))
        occ     = ', '.join(meta.get('style', {}).get('occasion_suitability', []))
        seasons = ', '.join(meta.get('seasonality', {}).get('season', []))
        lines.append(
            f'{iid} [{meta["category"]}] {meta.get("type","?")} | {colors} | {tags} | occ: {occ} | seasons: {seasons}'
        )
    wardrobe_text = '\n'.join(lines)

    user = (
        f'User request: occasion="{occasion}", season="{season}"\n\n'
        f'PRE-FILTERED WARDROBE ({len(lines)} items matching season/occasion):\n'
        f'{wardrobe_text}\n\n'
        f'{diversity_instruction()}\n'
        f'Use only item_ids from the list above.\n'
        f'Return JSON: {OUTFIT_JSON_SCHEMA}\n'
        f'Provide {NUM_RECS + 2} outfits (extra buffer); I will use the first {NUM_RECS} valid ones.'
    )

    resp = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'system', 'content': STYLIST_SYSTEM}, {'role': 'user', 'content': user}],
        temperature=0.4,
    )
    latency = time.perf_counter() - t0

    outfits = parse_json_response(resp.choices[0].message.content).get('outfits', [])
    outfits = validate_outfits(outfits, wardrobe)
    return {
        'recommendations': outfits,
        'latency_s': round(latency, 3),
        **calc_cost(resp.usage),
        'filter_size': len(lines),
    }

In [75]:
# ── P6: filtered_vision_llm ───────────────────────────────────────────────────
# Metadata pre-filter (season + occasion), then vision LLM sees filtered items' images.
# Isolates: within a semantically pre-filtered set, does seeing images help over text (P5)?

def run_p6_filtered_vision_llm(run: dict) -> dict:
    occasion, season = run['occasion'], run['season']

    t0 = time.perf_counter()
    filtered = filter_by_context(occasion, season)   # reuse P5's filter

    # Group by category and limit images sent (vision is expensive)
    by_cat: dict[str, list[str]] = {}
    for iid, meta in filtered.items():
        cat = meta.get('category', '')
        if cat in VALID_CATS:
            by_cat.setdefault(cat, []).append(iid)

    # Cap per-category to avoid token explosion (same as P4's VISION_CANDIDATES)
    for cat in by_cat:
        by_cat[cat] = by_cat[cat][:VISION_CANDIDATES]

    content: list[dict] = [{'type': 'text', 'text': (
        f'User request: occasion="{occasion}", season="{season}".\n'
        f'Below are pre-filtered wardrobe items (matched by season/occasion metadata).\n'
        f'Each image is labelled with its item_id.\n'
    )}]

    for cat, item_ids in by_cat.items():
        content.append({'type': 'text', 'text': f'\n### {cat.upper()}:'})
        for iid in item_ids:
            img_path = IMAGES_DIR / (iid + '.jpg')
            if not img_path.exists():
                continue
            content.append({
                'type': 'image_url',
                'image_url': {'url': f'data:image/jpeg;base64,{_encode_image(img_path)}'}
            })
            content.append({'type': 'text', 'text': f'id:{iid}'})

    content.append({'type': 'text', 'text': (
        f'\n{diversity_instruction()}\n'
        f'Use only item_ids shown in the labels above (e.g. id:182455006 → use 182455006).\n'
        f'Return JSON: {OUTFIT_JSON_SCHEMA}\n'
        f'Provide {NUM_RECS + 2} outfits (extra buffer); I will use the first {NUM_RECS} valid ones.'
    )})

    resp = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'system', 'content': STYLIST_SYSTEM}, {'role': 'user', 'content': content}],
        temperature=0.4,
    )
    latency = time.perf_counter() - t0

    outfits = parse_json_response(resp.choices[0].message.content).get('outfits', [])
    outfits = validate_outfits(outfits, wardrobe)
    return {
        'recommendations': outfits,
        'latency_s': round(latency, 3),
        **calc_cost(resp.usage),
        'filter_size': sum(len(v) for v in by_cat.values()),
    }


## 10. Run all pipelines + save results

In [76]:
FORCE_RERUN         = False  # set True to overwrite existing result files
PIPELINE_FNS = {
    'P1_text_llm':          run_p1_text_llm,
    'P2_clip_only':         run_p2_clip_only,
    'P3_clip_text_llm':     run_p3_clip_text_llm,
    'P4_clip_vision_llm':   run_p4_clip_vision_llm,
    'P5_filtered_text_llm': run_p5_filtered_text_llm,
    'P6_filtered_vision_llm': run_p6_filtered_vision_llm,
}

all_results = []

for run in runs:
    run_id  = run['run_id']
    out_dir = RESULTS_DIR / run_id
    out_dir.mkdir(exist_ok=True)
    print(f'\n══ Run: {run_id} (occasion={run["occasion"]}, season={run["season"]}) ══')

    for pipeline_id, fn in PIPELINE_FNS.items():
        out_file = out_dir / f'{pipeline_id}.json'

        if out_file.exists() and not FORCE_RERUN:
            print(f'  {pipeline_id}: already exists, skipping (set FORCE_RERUN=True to overwrite)')
            record = json.loads(out_file.read_text())
            all_results.append(record)
            continue

        print(f'  {pipeline_id}: running...', end=' ', flush=True)
        try:
            result = fn(run)
            record = {
                'run_id':    run_id,
                'pipeline':  pipeline_id,
                'prompt':    {'occasion': run['occasion'], 'season': run['season']},
                **result,
            }
            out_file.write_text(json.dumps(record, indent=2))
            all_results.append(record)
            print(f'done  latency={result["latency_s"]}s  cost=${result["cost_usd"]:.5f}  recs={len(result["recommendations"])}')
        except Exception as e:
            print(f'ERROR: {e}')
            raise

print('\nAll pipelines complete.')


══ Run: r1_office_winter (occasion=office wear, season=winter) ══
  P1_text_llm: already exists, skipping (set FORCE_RERUN=True to overwrite)
  P2_clip_only: running... done  latency=0.389s  cost=$0.00000  recs=5
  P3_clip_text_llm: already exists, skipping (set FORCE_RERUN=True to overwrite)
  P4_clip_vision_llm: already exists, skipping (set FORCE_RERUN=True to overwrite)
  P5_filtered_text_llm: already exists, skipping (set FORCE_RERUN=True to overwrite)
  P6_filtered_vision_llm: already exists, skipping (set FORCE_RERUN=True to overwrite)

══ Run: r2_party_summer (occasion=party wear, season=summer) ══
  P1_text_llm: running...   Warning: 181144813 is tops not bottoms — skipping
done  latency=3.467s  cost=$0.00552  recs=5
  P2_clip_only: running... done  latency=0.05s  cost=$0.00000  recs=5
  P3_clip_text_llm: already exists, skipping (set FORCE_RERUN=True to overwrite)
  P4_clip_vision_llm: already exists, skipping (set FORCE_RERUN=True to overwrite)
  P5_filtered_text_llm: alrea

## 11. Pipeline summary (cost + latency)

In [77]:
print(f'{"Pipeline":<25} {"Run":<22} {"Latency(s)":>10} {"In tokens":>10} {"Out tokens":>10} {"Cost($)":>10}')
print('-' * 95)
total_cost = 0.0
for r in all_results:
    print(
        f'{r["pipeline"]:<25} {r["run_id"]:<22} '
        f'{r["latency_s"]:>10.2f} {r["input_tokens"]:>10,} '
        f'{r["output_tokens"]:>10,} {r["cost_usd"]:>10.5f}'
    )
    total_cost += r['cost_usd']
print('-' * 95)
print(f'Total cost: ${total_cost:.5f}')

Pipeline                  Run                    Latency(s)  In tokens Out tokens    Cost($)
-----------------------------------------------------------------------------------------------
P1_text_llm               r1_office_winter             3.53     20,134        293    0.00547
P2_clip_only              r1_office_winter             0.39          0          0    0.00000
P3_clip_text_llm          r1_office_winter             2.65      2,391        293    0.00104
P4_clip_vision_llm        r1_office_winter            14.08     20,028        286    0.00544
P5_filtered_text_llm      r1_office_winter             2.88     15,401        293    0.00429
P6_filtered_vision_llm    r1_office_winter             8.61     20,030        271    0.00541
P1_text_llm               r2_party_summer              3.47     20,158        320    0.00552
P2_clip_only              r2_party_summer              0.05          0          0    0.00000
P3_clip_text_llm          r2_party_summer              2.25      2,

---
## 12. Post-annotation analysis

Run this section **after** annotating in `eval_v2/annotate.html` and exporting `human_scores.json`.

In [78]:
SCORES_FILE = ROOT / 'eval_v2' / 'human_scores.json'

if not SCORES_FILE.exists():
    print('human_scores.json not found — annotate first, then re-run this cell.')
else:
    scores = json.loads(SCORES_FILE.read_text())
    print(f'Loaded {len(scores)} scored combos')

    # Build score lookup: (run_id, pipeline, rec_index) -> score
    score_map = {(s['run_id'], s['pipeline'], s['rec_index']): s['score'] for s in scores}

    # Aggregate per pipeline
    from collections import defaultdict
    pipeline_scores: dict[str, list[int]] = defaultdict(list)
    for s in scores:
        pipeline_scores[s['pipeline']].append(s['score'])

    print(f'\n{"Pipeline":<25} {"N":>4} {"Avg Score":>10} {"≥3 (Approval%)":>15} {"Avg Latency":>12} {"Avg Cost($)":>12}')
    print('-' * 85)

    for pipeline_id in PIPELINE_FNS:
        sc   = pipeline_scores.get(pipeline_id, [])
        recs = [r for r in all_results if r['pipeline'] == pipeline_id]
        if not sc or not recs:
            continue
        avg_score    = sum(sc) / len(sc)
        approval_pct = 100 * sum(1 for s in sc if s >= 3) / len(sc)
        avg_latency  = sum(r['latency_s'] for r in recs) / len(recs)
        avg_cost     = sum(r['cost_usd']  for r in recs) / len(recs)
        print(
            f'{pipeline_id:<25} {len(sc):>4} {avg_score:>10.2f} '
            f'{approval_pct:>14.1f}% {avg_latency:>12.2f} {avg_cost:>12.5f}'
        )

    # Per-run breakdown
    print('\n── Per-run scores ──')
    for run in runs:
        print(f'\n{run["run_id"]} ({run["occasion"]}, {run["season"]}):')
        run_scores = [s for s in scores if s['run_id'] == run['run_id']]
        by_pipeline: dict[str, list[int]] = defaultdict(list)
        for s in run_scores:
            by_pipeline[s['pipeline']].append(s['score'])
        for pid, sc in sorted(by_pipeline.items()):
            print(f'  {pid:<25}  avg={sum(sc)/len(sc):.2f}  scores={sc}')

Loaded 59 scored combos

Pipeline                     N  Avg Score  ≥3 (Approval%)  Avg Latency  Avg Cost($)
-------------------------------------------------------------------------------------
P1_text_llm                 10       4.50           90.0%         3.50      0.00550
P2_clip_only                10       3.50           70.0%         0.22      0.00000
P3_clip_text_llm            10       3.80           90.0%         2.45      0.00098
P4_clip_vision_llm          10       3.60           80.0%        18.67      0.00543
P5_filtered_text_llm         9       4.22           88.9%         2.82      0.00459
P6_filtered_vision_llm      10       3.20           80.0%        14.08      0.00540

── Per-run scores ──

r1_office_winter (office wear, winter):
  P1_text_llm                avg=4.20  scores=[5, 5, 5, 1, 5]
  P2_clip_only               avg=4.60  scores=[5, 4, 5, 5, 4]
  P3_clip_text_llm           avg=4.80  scores=[5, 5, 5, 5, 4]
  P4_clip_vision_llm         avg=4.60  scores=[5, 4,